# Marmousi2 Acoustic bv1.2 Inversion Validation

This notebook validates the inversion side of the Marmousi2 acoustic case. It follows the original example style: parameters, model roles, survey subset, and source wavelet settings are defined directly in the notebook before running inversion.

## 1. Paths And Imports

The inversion uses synthetic-true observations generated by the current code, then starts from `init_model.npz`.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "ADFWI").exists():
    REPO_ROOT = Path("/liufeng1afs/project/04_Inversion/ADFWI-github")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ADFWI.survey import Receiver, Source, Survey
from ADFWI.utils import wavelet

CASE_DIR = REPO_ROOT / "examples" / "acoustic" / "01-model-test" / "01-Marmousi2"
VALIDATION_DIR = REPO_ROOT / "examples" / "validation" / "marmousi2_acoustic_bv12"
SCRIPT = VALIDATION_DIR / "scripts" / "run_validation.py"
OUTPUT_ROOT = VALIDATION_DIR / "outputs"

CASE_DIR


## 2. Local Case Definitions

These helpers are intentionally defined inside the notebook so this validation case does not import setup definitions from another example.

In [ ]:
def load_npz(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"required Marmousi2 file does not exist: {path}")
    return np.load(path, allow_pickle=True)


def cumulative_trapezoid(values: np.ndarray, dt: float) -> np.ndarray:
    out = np.zeros_like(values, dtype=np.float32)
    if values.size > 1:
        out[1:] = np.cumsum((values[:-1] + values[1:]) * (0.5 * dt), dtype=np.float64).astype(np.float32)
    return out


def build_source_wavelet(nt: int, dt: float, f0: float) -> np.ndarray:
    _, src_v = wavelet(nt, dt, f0, amp0=1)
    return cumulative_trapezoid(src_v.astype(np.float32), dt)


def build_survey(obs_npz, f0: float) -> Survey:
    nt = int(obs_npz["nt"])
    dt = float(obs_npz["dt"])
    src_loc = np.asarray(obs_npz["src_loc"], dtype=np.int64)
    rcv_loc = np.asarray(obs_npz["rcv_loc"], dtype=np.int64)
    src_type = np.asarray(obs_npz["src_type"]).astype(str)
    rcv_type = np.asarray(obs_npz["rcv_type"]).astype(str)

    src_v = build_source_wavelet(nt, dt, f0)
    source = Source(nt=nt, dt=dt, f0=f0)
    for (src_x, src_z), src_kind in zip(src_loc, src_type):
        source.add_source(int(src_x), int(src_z), src_v, src_type=str(src_kind))

    receiver = Receiver(nt=nt, dt=dt)
    for (rcv_x, rcv_z), rcv_kind in zip(rcv_loc, rcv_type):
        receiver.add_receiver(int(rcv_x), int(rcv_z), rcv_type=str(rcv_kind))
    return Survey(source, receiver)


## 3. Parameter Definitions

These values mirror the current short Marmousi2 bv1.2 validation path. The 100-iteration stage should only be run after inspecting the 10-iteration output.

In [ ]:
INVERSION_CONFIG = {
    "device": "npu:0",
    "dtype": "float32",
    "true_model_file": "true_model.npz",
    "inversion_model_file": "init_model.npz",
    "f0": 5.0,
    "shots": 3,
    "nt_samples": 3000,
    "checkpoint_segments": 10,
    "iterations_short": 10,
    "iterations_long": 100,
    "optimizer": "adam",
    "lr": 10.0,
    "scheduler_step_size": 200,
    "scheduler_gamma": 0.75,
    "misfit": "legacy-l2",
    "waveform_normalize": True,
    "auto_update_rho": True,
    "gradient_processor": "legacy",
}
INVERSION_CONFIG


## 4. Model Definitions

`true_model.npz` is used only to synthesize observed data. `init_model.npz` is the starting model for inversion.

In [ ]:
def model_summary(model_file: str):
    model_npz = load_npz(CASE_DIR / "data" / "model" / model_file)
    vp = np.asarray(model_npz["vp"])
    rho = np.asarray(model_npz["rho"])
    return {
        "model_file": model_file,
        "nx": int(model_npz["nx"]),
        "nz": int(model_npz["nz"]),
        "dx": float(model_npz["dx"]),
        "dz": float(model_npz["dz"]),
        "vp_shape": list(vp.shape),
        "vp_min": float(vp.min()),
        "vp_max": float(vp.max()),
        "rho_min": float(rho.min()),
        "rho_max": float(rho.max()),
    }

true_model = model_summary(INVERSION_CONFIG["true_model_file"])
initial_model = model_summary(INVERSION_CONFIG["inversion_model_file"])
true_model, initial_model


## 5. Observation System And Wavelet Definition

The survey and source wavelet are rebuilt from saved Marmousi2 observation metadata. The inversion stage uses the first `shots` sources and first `nt_samples` samples.

In [ ]:
obs_npz = load_npz(CASE_DIR / "data" / "waveform" / "obs_data.npz")
survey = build_survey(obs_npz, f0=INVERSION_CONFIG["f0"])
source_wavelet = build_source_wavelet(survey.source.nt, survey.source.dt, INVERSION_CONFIG["f0"])
survey_summary = {
    "full_shots": survey.source.num,
    "used_shots": INVERSION_CONFIG["shots"],
    "receivers": survey.receiver.num,
    "full_nt": survey.source.nt,
    "used_nt_samples": INVERSION_CONFIG["nt_samples"],
    "dt": survey.source.dt,
    "f0": INVERSION_CONFIG["f0"],
    "wavelet_norm": float(np.linalg.norm(source_wavelet)),
    "src_x_range": [int(np.min(survey.source.get_loc()[:, 0])), int(np.max(survey.source.get_loc()[:, 0]))],
    "rcv_x_range": [int(np.min(survey.receiver.get_loc()[:, 0])), int(np.max(survey.receiver.get_loc()[:, 0]))],
}
survey_summary


## 6. Run Inversion Validation

The default cell is a dry run. Set `RUN_INVERSION_10 = True` to execute the 10-iteration validation.

In [ ]:
def run_stage(stage: str, *, dry_run: bool = True, overwrite: bool = False, iterations: int | None = None):
    command = [
        sys.executable,
        str(SCRIPT),
        stage,
        "--case-dir", str(CASE_DIR),
        "--device", INVERSION_CONFIG["device"],
        "--dtype", INVERSION_CONFIG["dtype"],
        "--f0", str(INVERSION_CONFIG["f0"]),
        "--inversion-model-file", INVERSION_CONFIG["inversion_model_file"],
        "--shots", str(INVERSION_CONFIG["shots"]),
        "--nt-samples", str(INVERSION_CONFIG["nt_samples"]),
        "--checkpoint-segments", str(INVERSION_CONFIG["checkpoint_segments"]),
        "--lr", str(INVERSION_CONFIG["lr"]),
        "--scheduler-step-size", str(INVERSION_CONFIG["scheduler_step_size"]),
        "--scheduler-gamma", str(INVERSION_CONFIG["scheduler_gamma"]),
        "--gradient-processor", INVERSION_CONFIG["gradient_processor"],
        "--output-root", str(OUTPUT_ROOT),
    ]
    if iterations is not None:
        command.extend(["--iterations", str(iterations)])
    if dry_run:
        command.append("--dry-run")
    if overwrite:
        command.append("--overwrite")
    proc = subprocess.run(command, cwd=str(REPO_ROOT), text=True, capture_output=True, check=False)
    if proc.stderr:
        print(proc.stderr)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        raise RuntimeError(f"stage {stage} failed with return code {proc.returncode}")
    return json.loads(proc.stdout[proc.stdout.find("{"):])

run_stage("inversion10", dry_run=True, iterations=INVERSION_CONFIG["iterations_short"])


In [ ]:
RUN_INVERSION_10 = False

if RUN_INVERSION_10:
    inversion10_result = run_stage(
        "inversion10",
        dry_run=False,
        overwrite=True,
        iterations=INVERSION_CONFIG["iterations_short"],
    )
else:
    inversion10_result = {"status": "skipped", "reason": "set RUN_INVERSION_10=True"}

inversion10_result


## 7. Optional 100-Iteration Run

Only run this after the 10-iteration loss curve and model update are reasonable.

In [ ]:
RUN_INVERSION_100 = False

if RUN_INVERSION_100:
    inversion100_result = run_stage(
        "inversion100",
        dry_run=False,
        overwrite=True,
        iterations=INVERSION_CONFIG["iterations_long"],
    )
else:
    inversion100_result = {"status": "skipped", "reason": "set RUN_INVERSION_100=True"}

inversion100_result
